<a id="top"></a>
# Paper 2+ — `GeoFiLM + Canonical Closed-Loop Bridge` (Gold Run)

Notebook ini **mereplikasi persis resep `M4_GeoFiLM`** dari `01_GeoFiLM_Paper2_Master_Ablation.ipynb` (judge 84,30 — paritas dengan Gautam 84,52), lalu menambahkan **satu** komponen: kepala kanonik per kategori yang prediksinya **disuntikkan ke sisi encoder** (generalisasi CL-Bridge dari PolypSize).

**Rantai bukti yang mendasari desain** (semua dari data proyek ini):

| # | Temuan | Sumber |
|---|---|---|
| 1 | Informasi lokasi/ukuran/warna **ada** secara linier di fitur visual beku → masalahnya *readout*, bukan persepsi | notebook 04 (probe) |
| 2 | Slot sebagai **awalan keluaran** tidak pindah ke prosa: konsistensi prosa U4 terhadap slot-nya = setara kontrol. Judge hanya membaca prosa | analisis prediksi U4 vs M4 |
| 3 | Prediksi yang **disuntik ke encoder** pindah ke prosa: polyp_size 70,5 → 81,8; tiga kategori lokasi ikut naik | CL-Bridge (PolypSize) |
| 4 | 83% kekalahan M4 dari Gautam (−1,50 dari −1,81pp) ada di polyp_size + 3 kategori lokasi | CSV judge 18 kategori |
| 5 | Regresi U4 (−2,74pp token-F1) merata termasuk baris tanpa slot; tanda khas focal loss (kata di luar kosakata train 45× lebih sering) | analisis prediksi U4 |

**Yang dipertahankan identik dengan M4:** split, prompt, target (prosa murni, **tanpa slot**), loss VQA (CE bawaan, **tanpa focal**), LoRA r=32, HP, LR group, inisialisasi LoRA & kepala GeoFiLM, **urutan batch** (status RNG dipulihkan setelah membangun bridge), decoding greedy `max_new_tokens=64`.

**Yang ditambahkan:** `CanonicalBridge` — 16 kepala (satu per kategori ber-jawaban-tertutup, dipilih oleh **aturan data yang sama untuk semua kategori**, bukan daftar tangan), rumus injeksi CL-Bridge **utuh per kategori**, loss bantu α=0,05 dan LR ×100 (keduanya persis CL-Bridge), gerbang nol saat init (step 0 ≡ M4).

* **Folder output:** `experiments/Paper2plus_Bridge_r32`
* **Modul:** `GeoFiLM_Paper2/geofilm_bridge_core.py` (versi dikunci di `[A7]`)
* **Sakelar:** `VARIANT` di `[A9]` — `"B1_M4_CanonBridge"` (gold run) atau `"B0_M4_repro"` (kontrol replikasi M4)

### Urutan menjalankan
| Mode | Cell |
|---|---|
| **FRESH** (run baru) | `[A1]` → restart runtime → `[A2]`–`[A11]` → `[B1]` → `[D1]` → `[D2]` |
| **RESUME** (terputus) | `[A2]`–`[A10]` → `[C1]` → `[D1]` → `[D2]` |
| **EVAL** (setelah training) | `[A2]`–`[A9]` → `[E1]` → `[E2]` → `[E3]` → `[E4]` |

### Navigasi cepat
| Bagian | Isi |
|---|---|
| [PART A](#part-a) | Setup, data, label kanonik, arsitektur, smoke test |
| [PART B](#part-b) | Run FRESH |
| [PART C](#part-c) | RESUME |
| [PART D](#part-d) | Training loop (dengan pembanding kurva val M4 secara live) |
| [PART E](#part-e) | Inferensi, metrik live vs M4/A0, McNemar, diagnostik kepala |

# PART A — COMMON SETUP, LABEL KANONIK & ARSITEKTUR (Jalankan untuk FRESH, RESUME, maupun EVAL)
<a id="part-a"></a>

# [A1] Environment — install paket

In [1]:
# [A1] Environment — install paket & restart runtime sekali (identik notebook 01)
!pip install -q transformers==4.49.0 peft "torchao>=0.16.0" rouge-score sacrebleu nltk scikit-learn
import transformers, peft
print("transformers:", transformers.__version__, "| peft:", peft.__version__)
print("\n>>> JIKA BARU INSTAL PAKET, RESTART RUNTIME SEKARANG, lalu lanjutkan mulai dari Cell [A2] <<<")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
transformers: 4.49.0 | peft: 0.21.2

>>> JIKA BARU INSTAL PAKET, RESTART RUNTIME SEKARANG, lalu lanjutkan mulai dari Cell [A2] <<<


# [A2] Import & cek versi pustaka

In [1]:
# [A2] Import & cek versi pustaka
import transformers, peft, torch, gc
print("transformers:", transformers.__version__, "| peft:", peft.__version__, "| torch:", torch.__version__)
assert transformers.__version__ == "4.49.0", "Versi transformers harus 4.49.0 (sama dengan run M4)."

transformers: 4.49.0 | peft: 0.21.2 | torch: 2.11.0+cu130


# [A3] Mount Google Drive

In [2]:
# [A3] Mount Google Drive (otomatis dilewati jika berjalan di mesin lokal Windows/Linux)
import os
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    IS_COLAB = True
except ImportError:
    IS_COLAB = False
print("Running on Colab:", IS_COLAB)

Mounted at /content/drive
Running on Colab: True


# [A4] Konfigurasi path, seed, & caching gambar

In [3]:
# [A4] Konfigurasi path, seed, & caching gambar ke disk lokal (identik notebook 01, OUTDIR baru)
from pathlib import Path
import json, re, math, random, os, shutil, time
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F

if IS_COLAB:
    BASE = Path("/content/drive/MyDrive/DOKTORAL/KvasirVQAX1")
    LOCAL = Path("/content/all_images")
else:
    BASE = Path(r"g:/My Drive/DOKTORAL/KvasirVQAX1")
    LOCAL = BASE / "all_images"

SRC_DIR  = BASE / "all_images"
ZIP_PATH = BASE / "all_images.zip"

t0_img = time.time()
if IS_COLAB and (not LOCAL.exists() or len(os.listdir(LOCAL)) < 6449):
    LOCAL.mkdir(parents=True, exist_ok=True)
    if ZIP_PATH.exists():
        print(f"Menyalin arsip {ZIP_PATH.name} dari Google Drive ke disk lokal VM...")
        shutil.copy2(ZIP_PATH, "/content/all_images.zip")
        print("Mengekstrak gambar ke disk lokal (/content/all_images)...")
        os.system("unzip -q -o /content/all_images.zip -d /content/all_images")
        if os.path.exists("/content/all_images.zip"):
            os.remove("/content/all_images.zip")
        print(f"✓ Selesai ekstrak dalam {time.time() - t0_img:.1f} detik! Total gambar: {len(os.listdir(LOCAL))}")
    else:
        from tqdm.auto import tqdm
        src_files = [f for f in os.listdir(SRC_DIR) if not f.startswith(".")]
        loc_files = set(os.listdir(LOCAL))
        missing_files = [f for f in src_files if f not in loc_files]
        for f in tqdm(missing_files, desc="Salin Gambar", unit="file"):
            shutil.copy2(SRC_DIR / f, LOCAL / f)
        print(f"✓ Selesai salin dalam {time.time() - t0_img:.1f} detik! Total gambar: {len(os.listdir(LOCAL))}")
else:
    print(f"✓ Folder gambar siap: {LOCAL} ({len(os.listdir(LOCAL)) if LOCAL.exists() else 0} file).")

IMG_DIR    = LOCAL
A0_R32_DIR = BASE / "experiments/Stage2_CORAL_linear/prediction_rank32/20260916-060029_VQAonly-r32_a0.0_s42"
SPLIT_PATH = BASE / "experiments/Stage2_CORAL_linear/val_split_images.json"
BASE_ID    = "microsoft/Florence-2-base-ft"
CORE_DIR   = BASE / "GeoFiLM_Paper2"                       # lokasi geofilm_bridge_core.py
M4_DIR     = BASE / "experiments/Paper2_GeoFiLM_r32"       # run M4 (pembanding LIVE, bukan angka tulisan tangan)
OUTDIR     = BASE / "experiments/Paper2plus_Bridge_r32"
OUTDIR.mkdir(parents=True, exist_ok=True)

if SPLIT_PATH.exists() and not (OUTDIR / "val_split_images.json").exists():
    shutil.copy2(SPLIT_PATH, OUTDIR / "val_split_images.json")

for name, p in [("all_images", IMG_DIR), ("A0_vqaonly_r32", A0_R32_DIR), ("split", SPLIT_PATH),
                ("core_dir", CORE_DIR), ("m4_dir", M4_DIR)]:
    print("OK  " if p.exists() else "MISS", f"{p}")
print("OK   OUTDIR ->", OUTDIR)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
dev = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", dev, torch.cuda.get_device_name(0) if dev == "cuda" else "")

Menyalin arsip all_images.zip dari Google Drive ke disk lokal VM...
Mengekstrak gambar ke disk lokal (/content/all_images)...
✓ Selesai ekstrak dalam 11.1 detik! Total gambar: 6449
OK   /content/all_images
OK   /content/drive/MyDrive/DOKTORAL/KvasirVQAX1/experiments/Stage2_CORAL_linear/prediction_rank32/20260916-060029_VQAonly-r32_a0.0_s42
OK   /content/drive/MyDrive/DOKTORAL/KvasirVQAX1/experiments/Stage2_CORAL_linear/val_split_images.json
OK   /content/drive/MyDrive/DOKTORAL/KvasirVQAX1/GeoFiLM_Paper2
OK   /content/drive/MyDrive/DOKTORAL/KvasirVQAX1/experiments/Paper2_GeoFiLM_r32
OK   OUTDIR -> /content/drive/MyDrive/DOKTORAL/KvasirVQAX1/experiments/Paper2plus_Bridge_r32
device: cuda NVIDIA A100-SXM4-80GB


# [A5] Warisi Hyperparameter & konfigurasi LoRA r=32

In [4]:
# [A5] Warisi Hyperparameter & konfigurasi LoRA r=32 langsung dari Baseline A0 (identik notebook 01)
A0_CFG_PATH  = A0_R32_DIR / "config.json"
A0_LORA_PATH = A0_R32_DIR / "checkpoints/final/adapter_config.json"

if A0_CFG_PATH.exists():
    a0_meta = json.loads(A0_CFG_PATH.read_text())
    HP = a0_meta["hp"]
    print(f"Loaded HP from A0 VQA-only r=32 ({A0_R32_DIR.name}):")
else:
    HP = {
        "lr": 1e-05, "batch_size": 16, "grad_accum": 2, "epochs": 5.0,
        "warmup_ratio": 0.1, "weight_decay": 0.01, "scheduler": "linear",
        "fp16": True, "bf16": False
    }
    print("Loaded canonical A0 VQA-only r=32 HP:")

for k, v in HP.items():
    print(f"  {k:14s} {v}")

if A0_LORA_PATH.exists():
    lora_cfg = json.loads(A0_LORA_PATH.read_text())
else:
    lora_cfg = {
        "r": 32, "lora_alpha": 64, "lora_dropout": 0.05,
        "target_modules": ["q_proj", "qkv", "proj", "v_proj"],
        "bias": "none", "task_type": "SEQ_2_SEQ_LM"
    }

print("\nLoRA r=32 config (from A0 VQA-only r=32):")
for k in ["r", "lora_alpha", "lora_dropout", "target_modules", "bias", "task_type"]:
    print(f"  {k:16s} {lora_cfg.get(k)}")
assert int(lora_cfg["r"]) == 32 and int(lora_cfg["lora_alpha"]) == 64, f"Expected r=32, got {lora_cfg}"

Loaded HP from A0 VQA-only r=32 (20260916-060029_VQAonly-r32_a0.0_s42):
  lr             1e-05
  batch_size     16
  grad_accum     2
  epochs         5.0
  warmup_ratio   0.1
  weight_decay   0.01
  scheduler      linear
  fp16           True
  bf16           False

LoRA r=32 config (from A0 VQA-only r=32):
  r                32
  lora_alpha       64
  lora_dropout     0.05
  target_modules   ['qkv', 'q_proj', 'proj', 'v_proj']
  bias             none
  task_type        SEQ_2_SEQ_LM


# [A6] Muat dataset train & terapkan Image-Disjoint Validation Split

In [5]:
# [A6] Muat dataset train & terapkan Image-Disjoint Validation Split yang 100% identik dengan M4
from datasets import load_from_disk

ds_tr = load_from_disk(str(BASE / "train"))
tr = ds_tr.to_pandas()
print("Total train rows:", len(tr))

strip_jpg = lambda s: re.sub(r"\.(jpg|jpeg|png)$", "", str(s), flags=re.I)
tr["_img"] = tr["img_id"].map(strip_jpg)

assert SPLIT_PATH.exists(), f"Split file tidak ditemukan di {SPLIT_PATH}"
val_imgs = set(json.loads(SPLIT_PATH.read_text())["val_images"])
tr["_split"] = np.where(tr["_img"].isin(val_imgs), "val", "train")

overlap = set(tr.loc[tr._split == "train", "_img"]) & set(tr.loc[tr._split == "val", "_img"])
assert len(overlap) == 0, "Image overlap antara train dan val harus 0!"

for s in ["train", "val"]:
    g = tr[tr._split == s]
    print(f"  {s:5s} rows={len(g):7d}  unique_images={g._img.nunique():5d}")
# Run M4 memakai split ini: 129.023 train / 14.571 val
assert (tr._split == "train").sum() == 129023 and (tr._split == "val").sum() == 14571, \
    "Ukuran split berbeda dari run M4 — perbandingan tidak lagi apple-to-apple."

Total train rows: 143594
  train rows= 129023  unique_images= 5590
  val   rows=  14571  unique_images=  622


# [A7] Muat `geofilm_bridge_core.py`, gerbang versi, & label kanonik

In [6]:
# [A7] Muat geofilm_bridge_core.py, GERBANG VERSI, & bangun label kanonik dari split TRAIN saja
# Gerbang versi ada karena inferensi U4 sempat berjalan dengan modul LAMA (Drive belum
# tersinkron ke Colab) — tanpa gerbang ini, run 6 jam bisa jalan dengan bug yang sama.
import sys, importlib

EXPECTED_CORE_VERSION = "bridge-2026-11-03b"
assert (CORE_DIR / "geofilm_bridge_core.py").exists(), \
    f"geofilm_bridge_core.py tidak ada di {CORE_DIR}. Taruh file modul di folder GeoFiLM_Paper2."
if str(CORE_DIR) not in sys.path:
    sys.path.insert(0, str(CORE_DIR))
import geofilm_bridge_core as GB
importlib.reload(GB)
assert GB.CORE_VERSION == EXPECTED_CORE_VERSION, (
    f"Versi modul di Drive = {GB.CORE_VERSION!r}, seharusnya {EXPECTED_CORE_VERSION!r}.\n"
    "Drive belum tersinkron: tunggu 1-2 menit, Runtime > Restart, lalu ulangi dari [A2].")
GB.selfcheck_as_list()
from geofilm_bridge_core import (GeoFiLMModule, CanonicalBridge, GeoFiLMBridge, CanonLabeler,
                                 canonical_loss, canonical_predict, canonical_batch_stats,
                                 discover_canonical_specs, as_list, atoms, as_class_list,
                                 aspect_pairs)
print("✓ geofilm_bridge_core", GB.CORE_VERSION, "| as_list lulus uji format asli dataset\n")

# Aturan inklusi SAMA untuk semua kategori (tidak ada daftar kategori buatan tangan):
#   kosakata = atom berfrekuensi >= MIN_COUNT di split TRAIN; kategori ikut bila kosakata
#   menutupi >= MIN_COVER jawabannya dan punya >= 2 kemungkinan jawaban.
CANON_MIN_COUNT, CANON_MIN_COVER = 20, 0.99
CANON_SPECS = discover_canonical_specs(tr[tr._split == "train"],
                                       min_count=CANON_MIN_COUNT, min_cover=CANON_MIN_COVER)
LABELER = CanonLabeler(CANON_SPECS)

tr["_canon"] = [LABELER.encode(q, o) for q, o in zip(tr["question_class"], tr["original"])]

# Laporan cakupan: aspek yang ditanyakan vs aspek yang mendapat label sah (strict)
asked, got = {c: 0 for c in LABELER.cats}, {c: 0 for c in LABELER.cats}
for q, o, e in zip(tr["question_class"], tr["original"], tr["_canon"]):
    for c, _ in (aspect_pairs(q, o) or []):
        if c in asked:
            asked[c] += 1
    for ci, _ in e:
        got[LABELER.cats[ci]] += 1
cov = sum(got.values()) / max(sum(asked.values()), 1)
print(f"\nCakupan label kanonik (train+val): {sum(got.values())}/{sum(asked.values())} = {100*cov:.3f}%")
assert cov >= 0.99, "Cakupan label < 99% — periksa parsing `original`."
print(f"Rata-rata aspek berlabel per baris: {np.mean([len(e) for e in tr['_canon']]):.2f}")

print("\nSelf-test modul (identitas init = M4, gradien, RNG = M4, fp16 kasus terburuk):")
print(" ", GB.self_test(CANON_SPECS))

✓ geofilm_bridge_core bridge-2026-11-03b | as_list lulus uji format asli dataset

Spesifikasi kanonik (min_count=20, min_cover=0.99) | baris tak sejajar: 0
  kategori                   aspek  |V|  cover%  multi  status
  abnormality_color          14997   13  100.00   True  OK
  abnormality_location       15191    9   99.98   True  OK
  abnormality_presence       15013    5   99.96   True  OK
  box_artifact_presence      18954    2  100.00  False  OK
  finding_count              16525    4   99.87  False  OK
  finding_presence            2041    1  100.00  False  jawaban tunggal
  instrument_count           17241    4  100.00  False  OK
  instrument_location        15309   10  100.00   True  OK
  instrument_presence        15404    5   99.94   True  OK
  landmark_color               432    4   97.85   True  cover 0.978<0.99
  landmark_location          10759   10  100.00   True  OK
  landmark_presence           9761    4   99.85  False  OK
  polyp_count                15175    5   99.8

# [A8] Dataset `VQABridgeDS` & fungsi collate

In [7]:
# [A8] Dataset VQABridgeDS & fungsi collate (target VQA = prosa murni seperti M4 + label kanonik)
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from unittest.mock import patch
from transformers import AutoProcessor, AutoModelForCausalLM
from transformers.dynamic_module_utils import get_imports

PROMPT_PREFIX = "<VQA> "

def _skip_flash_proc(f):
    return [i for i in get_imports(f) if i != "flash_attn"]

with patch("transformers.dynamic_module_utils.get_imports", _skip_flash_proc):
    processor = AutoProcessor.from_pretrained(BASE_ID, trust_remote_code=True)

class VQABridgeDS(Dataset):
    def __init__(self, df):
        self.q = df["question"].astype(str).str.strip().tolist()
        self.a = df["answer"].astype(str).str.strip().tolist()     # TANPA slot: identik M4
        self.p = [IMG_DIR / f"{i}.jpg" for i in df["_img"]]
        self.c = df["_canon"].tolist()
    def __len__(self):
        return len(self.q)
    def __getitem__(self, i):
        p = self.p[i]
        try:
            img = Image.open(p).convert("RGB")
        except Exception:
            drive_src = BASE / "all_images" / p.name
            if drive_src.exists():
                shutil.copy2(drive_src, p)
                img = Image.open(p).convert("RGB")
            else:
                raise
        return {"q": PROMPT_PREFIX + self.q[i], "a": self.a[i], "img": img, "canon": self.c[i]}

def collate(batch):
    enc = processor(text=[b["q"] for b in batch],
                    images=[b["img"] for b in batch],
                    return_tensors="pt", padding=True)
    lab = processor.tokenizer([b["a"] for b in batch], return_tensors="pt",
                              padding=True, return_token_type_ids=False).input_ids
    lab[lab == processor.tokenizer.pad_token_id] = -100
    cy, cm = LABELER.to_tensors([b["canon"] for b in batch])
    return {
        "input_ids": enc["input_ids"],
        "pixel_values": enc["pixel_values"],
        "attention_mask": enc.get("attention_mask"),
        "labels": lab,
        "canon_y": cy,
        "canon_m": cm,
    }

ds_train = VQABridgeDS(tr[tr._split == "train"])
ds_val   = VQABridgeDS(tr[tr._split == "val"])
print(f"Dataset siap | Train: {len(ds_train)} sampel | Val: {len(ds_val)} sampel")

b_test = collate([ds_train[i] for i in range(4)])
bos_id = processor.tokenizer.bos_token_id
assert (b_test["labels"][:, 0] == bos_id).all(), "Token ke-0 pada labels harus bos_token_id!"
assert b_test["canon_y"].shape == (4, LABELER.T) and b_test["canon_m"].shape == (4, LABELER.C)
assert b_test["canon_m"].sum() > 0, "Batch uji tidak berisi label kanonik sama sekali."
# target VQA harus SAMA PERSIS dengan jawaban asli (tidak ada slot / prefiks apa pun)
dec = processor.tokenizer.batch_decode(b_test["labels"].clamp(min=0), skip_special_tokens=True)
_norm = lambda t: " ".join(str(t).split())
assert [_norm(d) for d in dec] == [_norm(a) for a in ds_train.a[:4]], "Target VQA tidak identik dengan jawaban asli!"
print(f"[VERIFIED] BOS ({bos_id}) | target = prosa asli | canon_y {tuple(b_test['canon_y'].shape)} "
      f"| aspek berlabel di batch uji: {int(b_test['canon_m'].sum())}")
del b_test, dec

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/806 [00:00<?, ?B/s]

processing_florence2.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/microsoft/Florence-2-base-ft:
- processing_florence2.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
/root/.cache/huggingface/modules/transformers_modules/microsoft/Florence-2-base-ft/f6c1a25888ffc1d945ee8a1a77ac833c7303d46e/processing_florence2.py:515: SyntaxWarning: invalid escape sequence '\d'
  PATTERN: 'r<time_(\d+)><time_(\d+)>([a-zA-Z0-9 ]+)'


tokenizer_config.json:   0%|          | 0.00/34.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

configuration_florence2.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/microsoft/Florence-2-base-ft:
- configuration_florence2.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


Dataset siap | Train: 129023 sampel | Val: 14571 sampel
[VERIFIED] BOS (0) | target = prosa asli | canon_y (4, 88) | aspek berlabel di batch uji: 11


# [A9] SAKELAR TUNGGAL `VARIANT`

In [8]:
# [A9] SAKELAR TUNGGAL VARIANT — gold run atau kontrol replikasi M4
#   "B1_M4_CanonBridge" -> GOLD RUN: kepala M4 persis + CanonicalBridge (16 kepala) + loss bantu
#   "B0_M4_repro"       -> Kontrol: kepala M4 persis tanpa bridge (memastikan pipeline = M4)
VARIANT = "B1_M4_CanonBridge"

M4_GEO_KWARGS = dict(use_geope=True, use_diff=True, gate_mode="film")   # = ABLATION_CONFIGS["M4_GeoFiLM"]
VARIANTS = {
    "B1_M4_CanonBridge": dict(use_bridge=True,  alpha_canon=0.05),
    "B0_M4_repro":       dict(use_bridge=False, alpha_canon=0.0),
}
assert VARIANT in VARIANTS, f"VARIANT tidak dikenal: {VARIANT}"
CFG = VARIANTS[VARIANT]
ALPHA_CANON = CFG["alpha_canon"]        # = alpha CL-Bridge (0,05) yang terbukti stabil
UNFREEZE_IMG_PROJ = True                # identik M4 & CL-Bridge
HEAD_FILE = "geofilm_bridge_head.pt"    # dipakai [D1] (simpan) dan [E2] (muat)

def _unwrap(m):
    """Cari modul yang mendefinisikan _encode_image (dipakai [A10] dan [E2])."""
    seen = set()
    def walk(x, depth=0):
        if depth > 6 or id(x) in seen: return None
        seen.add(id(x))
        if "_encode_image" in type(x).__dict__: return x
        for name in ("base_model", "model"):
            child = getattr(x, name, None)
            if child is not None and child is not x:
                r = walk(child, depth + 1)
                if r is not None: return r
        return None
    found = walk(m)
    assert found is not None, "could not locate the module defining _encode_image"
    return found

def make_head(d_vis=768):
    return GeoFiLMBridge(d_vis, CANON_SPECS, M4_GEO_KWARGS, use_bridge=CFG["use_bridge"]).to(dev)

print("Active VARIANT :", VARIANT, "|", CFG)
print("Geo (M4)       :", M4_GEO_KWARGS)
print("Kategori bridge:", len(CANON_SPECS), "| atom:", LABELER.T)

Active VARIANT : B1_M4_CanonBridge | {'use_bridge': True, 'alpha_canon': 0.05}
Geo (M4)       : {'use_geope': True, 'use_diff': True, 'gate_mode': 'film'}
Kategori bridge: 16 | atom: 88


# [A10] Build model Florence-2 + LoRA r=32 + hook

In [9]:
# [A10] Build model Florence-2 + LoRA r=32 + pasang hook get_input_embeddings & _encode_image
from peft import LoraConfig, get_peft_model
import transformers.dynamic_module_utils as dmu
orig_get_imports = dmu.get_imports

def _skip_flash(name, *args, **kwargs):
    if "flash_attn" in str(name):
        return None
    return orig_get_imports(name, *args, **kwargs)

LORA_R     = 32
LORA_ALPHA = LORA_R * 2

def build_model():
    with patch("transformers.dynamic_module_utils.get_imports", _skip_flash):
        m = AutoModelForCausalLM.from_pretrained(
            BASE_ID, trust_remote_code=True, torch_dtype=torch.float32)
    cfg = LoraConfig(
        r=LORA_R, lora_alpha=LORA_ALPHA,
        lora_dropout=lora_cfg["lora_dropout"],
        target_modules=list(lora_cfg["target_modules"]),
        bias=lora_cfg.get("bias", "none"),
        task_type=lora_cfg.get("task_type"))
    m = get_peft_model(m, cfg)

    core = _unwrap(m)
    if UNFREEZE_IMG_PROJ:
        core.image_projection.requires_grad_(True)
        for p in core.image_proj_norm.parameters():
            p.requires_grad_(True)

    cache = {"query_embeds": None, "query_mask": None, "reg_loss": None, "canon_logits": None}
    head_holder = [None]

    emb_layer = core.get_input_embeddings()
    def _emb_hook(module, args, output):
        if torch.is_tensor(output) and output.dim() == 3 and output.shape[1] > 2:
            cache["query_embeds"] = output.detach()
    emb_hook_handle = emb_layer.register_forward_hook(_emb_hook)

    orig_encode = core._encode_image
    def wrapped_encode_image(pixel_values):
        out = orig_encode(pixel_values)
        h_mod = head_holder[0]
        if h_mod is not None:
            bridged, reg_loss, canon_logits = h_mod.forward_bridge(
                out, query_embeds=cache.get("query_embeds"), query_mask=cache.get("query_mask"))
            cache["feats"] = out
            cache["reg_loss"] = reg_loss
            cache["canon_logits"] = canon_logits
            return bridged.to(out.dtype)
        cache["feats"] = out
        cache["reg_loss"] = torch.tensor(0.0, device=out.device)
        cache["canon_logits"] = None
        return out

    core._encode_image = wrapped_encode_image
    m._feat_cache = cache
    object.__setattr__(m, "_core_ref", [core])
    object.__setattr__(m, "_head_ref", head_holder)
    object.__setattr__(m, "_emb_hook_ref", [emb_hook_handle])
    return m.to(dev)

model = build_model()
core = model._core_ref[0]
D_VIS = core.image_projection.shape[1]
head = make_head(D_VIS)
model._head_ref[0] = head

model.print_trainable_parameters()
n_geo = sum(p.numel() for p in head.geo.parameters())
n_br  = sum(p.numel() for p in head.bridge.parameters()) if head.bridge is not None else 0
print(f"Kepala GeoFiLM (M4) : {n_geo:,} param")
print(f"CanonicalBridge     : {n_br:,} param  (+{100*n_br/max(n_geo,1):.1f}% dari kepala M4)")

REG_WEIGHT = 0.02                                                  # identik M4
# LR tiap komponen = LR yang SUDAH terbukti untuk komponen itu:
#   LoRA & kepala GeoFiLM -> persis run M4 (x50);  bridge -> persis CL-Bridge (x100, config run E).
LR_GROUPS  = {"lora": HP["lr"], "head": HP["lr"] * 50, "bridge": HP["lr"] * 100,
              "head_lr_mult": 50, "bridge_lr_mult": 100}
print("REG_WEIGHT:", REG_WEIGHT, "| ALPHA_CANON:", ALPHA_CANON, "| LR_GROUPS:", LR_GROUPS)

modeling_florence2.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/microsoft/Florence-2-base-ft:
- modeling_florence2.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors:   0%|          | 0.00/463M [00:00<?, ?B/s]

trainable params: 5,191,776 || all params: 235,817,824 || trainable%: 2.2016
Kepala GeoFiLM (M4) : 1,730,850 param
CanonicalBridge     : 237,064 param  (+13.7% dari kepala M4)
REG_WEIGHT: 0.02 | ALPHA_CANON: 0.05 | LR_GROUPS: {'lora': 1e-05, 'head': 0.0005, 'bridge': 0.001, 'head_lr_mult': 50, 'bridge_lr_mult': 100}


# SMOKE TEST — Uji Pra-Training (urutan training SEBENARNYA, beberapa langkah)

## [A11] Smoke Test Pra-Training

In [10]:
# [A11] Smoke Test Pra-Training — urutan [D2] persis (scale->backward->unscale->clip->step->update)
# Menilai gradien SEBELUM unscale itu salah (step pertama fp16 memang sering overflow & dilewati
# GradScaler — perilaku normal). Yang diuji: minimal satu step benar-benar diterapkan.
print("=" * 72)
print(f" SMOKE TEST PRA-TRAINING | VARIANT = {VARIANT}")
print("=" * 72)

SMOKE_STEPS, SMOKE_BS = 3, 2
model.train(); head.train()
model.zero_grad(set_to_none=True); head.zero_grad(set_to_none=True)
model._feat_cache.clear()
use_fp16 = HP.get("fp16", True) and (dev == "cuda")
smk_scaler = torch.amp.GradScaler("cuda", enabled=use_fp16)
smk_opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad] + list(head.parameters()), lr=1e-6)

steps_applied, identity_ok = 0, None
for st in range(SMOKE_STEPS):
    bb = collate([ds_train[st * SMOKE_BS + j] for j in range(SMOKE_BS)])
    bb = {k: (v.to(dev, non_blocking=True) if torch.is_tensor(v) else v) for k, v in bb.items()}
    model._feat_cache["query_mask"] = bb["attention_mask"]
    with torch.amp.autocast("cuda", enabled=use_fp16):
        o = model(input_ids=bb["input_ids"], pixel_values=bb["pixel_values"],
                  attention_mask=bb["attention_mask"], labels=bb["labels"])
        rl = model._feat_cache.get("reg_loss")
        lg = model._feat_cache.get("canon_logits")
        lc, n_lab = (canonical_loss(lg, bb["canon_y"], bb["canon_m"], LABELER) if lg is not None
                     else (torch.tensor(0.0, device=dev), 0))
        tot = o.loss + REG_WEIGHT * rl + ALPHA_CANON * lc
    assert torch.isfinite(tot), f"Loss NaN/Inf pada step {st} — BUG model, bukan skala fp16."
    assert model._feat_cache.get("query_embeds") is not None, "Hook query embedding tidak terpanggil!"
    if CFG["use_bridge"]:
        assert lg is not None and lg.shape == (SMOKE_BS, LABELER.T), f"canon_logits {None if lg is None else lg.shape}"

    if st == 0:
        # Identitas: sebelum step pertama, keluaran bridge HARUS sama persis dengan keluaran M4
        with torch.no_grad():
            f0 = model._feat_cache["feats"]
            qe, qm = model._feat_cache.get("query_embeds"), model._feat_cache.get("query_mask")
            b_out, _, _ = head.forward_bridge(f0, qe, qm)
            g_out, _ = head.geo.forward_bridge(f0, qe, qm)
            identity_ok = bool(torch.equal(b_out, g_out))
        assert identity_ok, "Step 0 tidak identik dengan M4 — gerbang bridge tidak nol?"

    sc0 = smk_scaler.get_scale()
    smk_scaler.scale(tot).backward()
    if CFG["use_bridge"]:
        assert head.bridge.gate_global.grad is not None and head.bridge.fc[0].weight.grad is not None, \
            "Bridge tidak menerima gradien."
    smk_scaler.unscale_(smk_opt)
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    torch.nn.utils.clip_grad_norm_(head.geo.parameters(), max_norm=1.0)
    if head.bridge is not None:
        torch.nn.utils.clip_grad_norm_(head.bridge.parameters(), max_norm=1.0)
    smk_scaler.step(smk_opt); smk_scaler.update()
    applied = smk_scaler.get_scale() >= sc0
    steps_applied += int(applied)
    print(f"  step {st}: L_VQA {float(o.loss):.4f} | L_struct {float(rl):.4f} | L_canon {float(lc):.4f} "
          f"(n={n_lab}) | {'DITERAPKAN' if applied else 'dilewati (overflow fp16 awal, normal)'}")
    smk_opt.zero_grad(set_to_none=True); model.zero_grad(set_to_none=True); head.zero_grad(set_to_none=True)
    model._feat_cache.clear()
    del o, rl, lg, lc, tot, bb

assert steps_applied > 0, "Tidak ada optimizer step yang diterapkan — gradien inf/nan persisten (bug)."
print(f" ✓ Identitas step 0 = M4 : {identity_ok}")
print(f" ✓ Trainability          : {steps_applied}/{SMOKE_STEPS} step diterapkan")
del smk_opt, smk_scaler
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("=" * 72)
print(" ✓ SMOKE TEST LULUS. Lanjut ke [B1] (model dibangun ulang bersih di sana).")
print("=" * 72)

 SMOKE TEST PRA-TRAINING | VARIANT = B1_M4_CanonBridge


/tmp/ipykernel_1025/2368258527.py:57: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  print(f"  step {st}: L_VQA {float(o.loss):.4f} | L_struct {float(rl):.4f} | L_canon {float(lc):.4f} "


  step 0: L_VQA 6.0754 | L_struct 0.0215 | L_canon 2.0213 (n=5) | dilewati (overflow fp16 awal, normal)
  step 1: L_VQA 4.3001 | L_struct 0.0215 | L_canon 4.2971 (n=6) | DITERAPKAN
  step 2: L_VQA 5.7993 | L_struct 0.0215 | L_canon 1.3007 (n=3) | DITERAPKAN
 ✓ Identitas step 0 = M4 : True
 ✓ Trainability          : 2/3 step diterapkan
 ✓ SMOKE TEST LULUS. Lanjut ke [B1] (model dibangun ulang bersih di sana).


# PART B — FRESH RUN SETUP (Jalankan untuk Memulai Run Baru dari Step 0)
<a id="part-b"></a>

## [B1] Reseed + rebuild model

In [11]:
# [B1] Reseed + rebuild model bersih untuk run FRESH (urutan RNG identik run M4)
from datetime import datetime

if 'model' in globals():
    model.zero_grad(set_to_none=True)
    if hasattr(model, '_feat_cache'):
        model._feat_cache.clear()
    if hasattr(model, '_emb_hook_ref'):
        for h in model._emb_hook_ref:
            try: h.remove()
            except Exception: pass
if 'head' in globals():
    head.zero_grad(set_to_none=True)
model = head = None
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

model = build_model()
core = model._core_ref[0]
D_VIS = core.image_projection.shape[1]
head = make_head(D_VIS)          # bridge memulihkan RNG -> init M4 & urutan batch = run M4
model._head_ref[0] = head

total_vram = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 16.0
EFFECTIVE_BS = HP["batch_size"] * HP["grad_accum"]
MICRO_BS     = 16 if (torch.cuda.is_available() and total_vram >= 30.0) else 4
GRAD_ACCUM   = EFFECTIVE_BS // MICRO_BS
assert MICRO_BS * GRAD_ACCUM == EFFECTIVE_BS
if MICRO_BS != 16:
    print("PERINGATAN: run M4 memakai micro-batch 16. Dengan micro-batch lain, subset validasi berbeda "
          "dan pembanding kurva val M4 di [D1] tidak lagi sebanding.")

RUN_TAG  = datetime.now().strftime("%Y%m%d-%H%M%S")
RUN_NAME = f"{RUN_TAG}_{VARIANT}_r{LORA_R}_s{SEED}"
RUN_DIR  = OUTDIR / RUN_NAME
(RUN_DIR / "checkpoints").mkdir(parents=True, exist_ok=True)
(RUN_DIR / "history").mkdir(parents=True, exist_ok=True)

CONFIG = {
    "run_name": RUN_NAME,
    "started_at": datetime.now().isoformat(timespec="seconds"),
    "variant": VARIANT,
    "variant_cfg": CFG,
    "geo_kwargs": M4_GEO_KWARGS,
    "core_version": GB.CORE_VERSION,
    "canon_min_count": CANON_MIN_COUNT,
    "canon_min_cover": CANON_MIN_COVER,
    "canon_specs": CANON_SPECS,
    "alpha_canon": ALPHA_CANON,
    "seed": SEED,
    "reg_weight": REG_WEIGHT,
    "lr_groups": LR_GROUPS,
    "hp": HP,
    "effective_batch": EFFECTIVE_BS,
    "micro_batch": MICRO_BS,
    "grad_accum": GRAD_ACCUM,
    "lora_r": LORA_R,
    "lora_alpha": LORA_ALPHA,
    "n_head_params": int(sum(p.numel() for p in head.parameters())),
    "unfreeze_image_projection": UNFREEZE_IMG_PROJ,
    "loss_vqa": "cross_entropy (out.loss bawaan, tanpa focal)",
    "target": "prosa asli (tanpa slot)",
}
(RUN_DIR / "config.json").write_text(json.dumps(CONFIG, indent=2))
(RUN_DIR / "canon_specs.json").write_text(json.dumps(CANON_SPECS, indent=1))
print("✓ Fresh model built & RUN_DIR created ->", RUN_DIR)
print(f"  Effective batch = {EFFECTIVE_BS} (micro {MICRO_BS} x accum {GRAD_ACCUM})")

✓ Fresh model built & RUN_DIR created -> /content/drive/MyDrive/DOKTORAL/KvasirVQAX1/experiments/Paper2plus_Bridge_r32/20261103-050623_B1_M4_CanonBridge_r32_s42
  Effective batch = 32 (micro 16 x accum 2)


# PART C — RESUME ONLY (Jalankan HANYA Jika Training Terputus di Tengah Jalan)
<a id="part-c"></a>

## [C1] Cari & arahkan ke folder run yang terputus

In [ ]:
# [C1] Cari & arahkan ke folder run yang terputus sesuai VARIANT
cands = []
for p in sorted(OUTDIR.glob("*")):
    if p.is_dir() and (p / "resume/state.pt").exists() and (p / "resume/adapter_model.safetensors").exists():
        d = torch.load(p / "resume/state.pt", map_location="cpu", weights_only=False)
        cands.append(p)
        print(p.name, "| step", d["global_step"], "| epoch", d["epoch"])
assert cands, f"Tidak ada run dengan resume slot di {OUTDIR}"

match = [p for p in cands if VARIANT in p.name]
RUN_DIR = match[-1] if match else cands[-1]
CONFIG = json.loads((RUN_DIR / "config.json").read_text())
assert CONFIG.get("core_version") == GB.CORE_VERSION, \
    f"Run dilatih dengan core {CONFIG.get('core_version')}, modul sekarang {GB.CORE_VERSION}"
assert list(CONFIG["canon_specs"].items()) == list(json.loads(json.dumps(CANON_SPECS)).items()), \
    "Spesifikasi kanonik (isi ATAU urutan) berbeda dari run yang di-resume — kepala tidak akan cocok."
MICRO_BS, GRAD_ACCUM = CONFIG["micro_batch"], CONFIG["grad_accum"]
print("\nRESUME TARGET ->", RUN_DIR.name)

# PART D — TRAINING LOOP UTAMA (5 Epoch)
<a id="part-d"></a>

Setiap validasi mencetak **selisih `val_L_VQA` terhadap run M4 pada step yang sama**, dibaca langsung dari `history/val.jsonl` run M4. Loss VQA di sini adalah CE yang sama persis dengan M4 (target prosa yang sama, subset val yang sama), sehingga angkanya **langsung sebanding**:
* step 0 harus ≈ nilai M4 (5,0737) — bukti pipeline identik;
* selisih negatif yang konsisten setelah ~2.000 step = sinyal awal bahwa run ini di jalur yang benar.

## [D1] Setup DataLoaders, Optimizer, Scheduler, Checkpointing, & Validasi

In [12]:
# [D1] Setup DataLoaders, Optimizer, Scheduler, Checkpointing, & Validasi (VQA + kanonik + pembanding M4)
from transformers import get_scheduler
from tqdm.auto import tqdm
import collections
from datetime import datetime

TRAIN_LOG  = RUN_DIR / "history/train.jsonl"
VAL_LOG    = RUN_DIR / "history/val.jsonl"
CKPT_DIR   = RUN_DIR / "checkpoints"
RESUME_DIR = RUN_DIR / "resume"
RESUME_DIR.mkdir(parents=True, exist_ok=True)

def log_line(path, d):
    with open(path, "a") as f:
        f.write(json.dumps(d) + "\n")

def save_ckpt(step, tag=None):
    td = CKPT_DIR / (tag if tag else f"step-{step:06d}")
    td.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(str(td))
    torch.save(head.state_dict(), td / HEAD_FILE)
    (td / "canon_specs.json").write_text(json.dumps(CANON_SPECS, indent=1))
    core = model._core_ref[0]
    torch.save({"image_projection": core.image_projection.data.clone(),
                "image_proj_norm":  core.image_proj_norm.state_dict()}, td / "image_proj.pt")
    print(f"saved checkpoint -> {td.name}")

def save_resume(step, epoch):
    tmp = RESUME_DIR / "state.pt.tmp"
    torch.save({
        "global_step": step, "epoch": epoch,
        "optimizer": opt.state_dict(), "scheduler": sched.state_dict(), "scaler": scaler.state_dict(),
        "rng": {"python": random.getstate(), "numpy": np.random.get_state(),
                "torch": torch.get_rng_state(),
                "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None},
        "head_state": head.state_dict(),
        "saved_at": datetime.now().isoformat(timespec="seconds"),
    }, tmp)
    os.replace(tmp, RESUME_DIR / "state.pt")
    model.save_pretrained(str(RESUME_DIR))
    core = model._core_ref[0]
    torch.save({"image_projection": core.image_projection.data.clone(),
                "image_proj_norm":  core.image_proj_norm.state_dict()}, RESUME_DIR / "image_proj.pt")

dl_tr = DataLoader(ds_train, batch_size=MICRO_BS, shuffle=True,
                   collate_fn=collate, num_workers=4 if IS_COLAB else 0, pin_memory=True)
dl_va = DataLoader(ds_val, batch_size=MICRO_BS, shuffle=False,
                   collate_fn=collate, num_workers=2 if IS_COLAB else 0)

# Grup optimizer: LoRA & kepala GeoFiLM identik M4; bridge memakai LR CL-Bridge (x100).
_groups = [
    {"params": [p for p in model.parameters() if p.requires_grad],
     "lr": LR_GROUPS["lora"], "weight_decay": HP["weight_decay"]},
    {"params": list(head.geo.parameters()),
     "lr": LR_GROUPS["head"], "weight_decay": HP["weight_decay"]},
]
if head.bridge is not None:
    _groups.append({"params": list(head.bridge.parameters()),
                    "lr": LR_GROUPS["bridge"], "weight_decay": HP["weight_decay"]})
opt = torch.optim.AdamW(_groups)
_n_opt = sum(p.numel() for g in opt.param_groups for p in g["params"])
_n_all = sum(p.numel() for p in model.parameters() if p.requires_grad) + sum(p.numel() for p in head.parameters())
assert _n_opt == _n_all, f"Optimizer tidak mencakup semua parameter terlatih ({_n_opt} vs {_n_all})"

STEPS_PER_EPOCH = len(dl_tr) // GRAD_ACCUM
TOTAL_STEPS     = int(STEPS_PER_EPOCH * HP["epochs"])
WARMUP_STEPS    = int(TOTAL_STEPS * HP["warmup_ratio"])
print(f"steps/epoch: {STEPS_PER_EPOCH} | total: {TOTAL_STEPS} | warmup: {WARMUP_STEPS}")

sched = get_scheduler(HP["scheduler"], optimizer=opt,
                      num_warmup_steps=WARMUP_STEPS, num_training_steps=TOTAL_STEPS)
scaler = torch.amp.GradScaler("cuda", enabled=HP["fp16"] and (dev == "cuda"))

def clip_all():
    # Kepala M4 di-clip SENDIRI (persis seperti run M4). Bila geo & bridge di-clip bersama,
    # gradien loss bantu bisa memperkecil langkah kepala M4 -> dinamika M4 berubah.
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    torch.nn.utils.clip_grad_norm_(head.geo.parameters(), max_norm=1.0)
    if head.bridge is not None:
        torch.nn.utils.clip_grad_norm_(head.bridge.parameters(), max_norm=1.0)

# Kurva val M4 dibaca LIVE dari run M4 yang lengkap (tidak ada angka yang ditulis tangan)
M4_VAL, M4_REF_RUN = {}, None
_m4 = [p for p in sorted(M4_DIR.glob("*M4_GeoFiLM*"))
       if (p / "history/val.jsonl").exists() and (p / "checkpoints/final").exists()]
if _m4:
    M4_REF_RUN = _m4[-1]
    for line in (M4_REF_RUN / "history/val.jsonl").read_text().splitlines():
        r = json.loads(line)
        M4_VAL[r["global_step"]] = r["val_vqa_loss"]
print("Pembanding kurva val:", M4_REF_RUN.name if M4_REF_RUN else "TIDAK DITEMUKAN", f"({len(M4_VAL)} titik)")

def validate_vqa(step, epoch, max_batches=60):
    model.eval(); head.eval()
    v_losses, r_losses, c_losses = [], [], []
    cstat = collections.defaultdict(lambda: [0, 0])
    with torch.no_grad():
        for k, bb in enumerate(dl_va):
            if max_batches and k >= max_batches:
                break
            bb = {kk: (v.to(dev, non_blocking=True) if torch.is_tensor(v) else v) for kk, v in bb.items()}
            model._feat_cache["query_mask"] = bb.get("attention_mask")
            with torch.amp.autocast("cuda", enabled=HP["fp16"] and (dev == "cuda")):
                o = model(input_ids=bb["input_ids"], pixel_values=bb["pixel_values"],
                          attention_mask=bb.get("attention_mask"), labels=bb["labels"])
                rl = model._feat_cache.get("reg_loss")
                lg = model._feat_cache.get("canon_logits")
            v_losses.append(float(o.loss.item()))
            if rl is not None:
                r_losses.append(float(rl.item()))
            if lg is not None:
                lc, _ = canonical_loss(lg, bb["canon_y"], bb["canon_m"], LABELER)
                c_losses.append(float(lc))
                for c, (n, ok) in canonical_batch_stats(lg, bb["canon_y"], bb["canon_m"], LABELER).items():
                    cstat[c][0] += n; cstat[c][1] += ok
            model._feat_cache.clear()
    model.train(); head.train()
    gg, gs = head.gate_mean_abs()
    acc_macro = (float(np.mean([ok / n for n, ok in cstat.values() if n])) if cstat else None)
    rec = {
        "global_step": step, "epoch": epoch,
        "val_vqa_loss": round(float(np.mean(v_losses)), 4) if v_losses else None,
        "val_reg_loss": round(float(np.mean(r_losses)), 4) if r_losses else None,
        "val_canon_loss": round(float(np.mean(c_losses)), 4) if c_losses else None,
        "val_canon_acc_macro": round(acc_macro, 4) if acc_macro is not None else None,
        "m4_val_vqa_loss": M4_VAL.get(step),
        "gate_global_absmean": gg, "gate_spatial_absmean": gs,
        "pe_scale": round(float(torch.tanh(head.pe_scale).item()), 4),
        "at": datetime.now().isoformat(timespec="seconds"),
    }
    log_line(VAL_LOG, rec)
    m4 = rec["m4_val_vqa_loss"]
    vs = (f" | M4 {m4:.4f} (Δ {rec['val_vqa_loss'] - m4:+.4f})" if m4 is not None else "")
    ca = (f" | canon acc {100*acc_macro:.1f}%" if acc_macro is not None else "")
    print(f"\n[VAL step {step} ep {epoch}] val_L_VQA={rec['val_vqa_loss']:.4f}{vs}{ca} "
          f"| gate |g| {gg:.4f}/{gs:.4f} | pe_scale={rec['pe_scale']:.4f}")
    if step == 0 and m4 is not None and abs(rec["val_vqa_loss"] - m4) > 0.02:
        print("  PERINGATAN: val step 0 berbeda dari M4 > 0,02 — pipeline mungkin tidak identik "
              "(cek micro-batch, split, prompt). Hentikan & laporkan sebelum melanjutkan.")
    return rec

steps/epoch: 4032 | total: 20160 | warmup: 2016
Pembanding kurva val: 20261029-134314_M4_GeoFiLM_r32_s42 (46 titik)


## [D2] Jalankan Loop Training 5 Epoch

In [13]:
# [D2] Jalankan Loop Training 5 Epoch (dengan Auto-Restore saat Resume & penjaga divergensi)
LOG_EVERY  = 50
VAL_EVERY  = 500
SAVE_EVERY = 500

gstep = 0
start_epoch = 0

resume_state = RUN_DIR / "resume/state.pt"
if resume_state.exists():
    d = torch.load(resume_state, map_location="cpu", weights_only=False)
    gstep = d["global_step"]
    start_epoch = d["epoch"]
    opt.load_state_dict(d["optimizer"])
    sched.load_state_dict(d["scheduler"])
    scaler.load_state_dict(d["scaler"])
    random.setstate(d["rng"]["python"])
    np.random.set_state(d["rng"]["numpy"])
    torch.set_rng_state(d["rng"]["torch"])
    if torch.cuda.is_available() and d["rng"]["cuda"] is not None:
        torch.cuda.set_rng_state_all(d["rng"]["cuda"])
    head.load_state_dict(d["head_state"])
    from peft import set_peft_model_state_dict
    from safetensors.torch import load_file
    set_peft_model_state_dict(model, load_file(str(RUN_DIR / "resume/adapter_model.safetensors")))
    if (RUN_DIR / "resume/image_proj.pt").exists():
        ip_st = torch.load(RUN_DIR / "resume/image_proj.pt", map_location=dev)
        core = model._core_ref[0]
        core.image_projection.data.copy_(ip_st["image_projection"])
        core.image_proj_norm.load_state_dict(ip_st["image_proj_norm"])
    print(f"✓ Resumed dari step {gstep}, epoch {start_epoch}")

t0 = time.time()
validate_vqa(gstep, start_epoch, max_batches=40)       # jadwal max_batches identik M4 (40/50/100)

DIVERGED = False
for ep in range(start_epoch, int(math.ceil(HP["epochs"]))):
    if DIVERGED:
        break
    model.train(); head.train()
    pbar = tqdm(dl_tr, desc=f"{VARIANT} epoch {ep}")
    for i, bb in enumerate(pbar):
        bb = {k: (v.to(dev, non_blocking=True) if torch.is_tensor(v) else v) for k, v in bb.items()}
        model._feat_cache["query_mask"] = bb.get("attention_mask")
        with torch.amp.autocast("cuda", enabled=HP["fp16"] and (dev == "cuda")):
            o = model(input_ids=bb["input_ids"], pixel_values=bb["pixel_values"],
                      attention_mask=bb.get("attention_mask"), labels=bb["labels"])
            rl = model._feat_cache.get("reg_loss")
            if rl is None:
                rl = torch.tensor(0.0, device=dev)
            lg = model._feat_cache.get("canon_logits")
            if lg is not None:
                lc, n_lab = canonical_loss(lg, bb["canon_y"], bb["canon_m"], LABELER)
            else:
                lc, n_lab = torch.tensor(0.0, device=dev), 0
            # Loss VQA = CE bawaan (identik M4). Loss bantu kanonik ditambahkan dengan alpha CL-Bridge.
            loss = (o.loss + REG_WEIGHT * rl + ALPHA_CANON * lc) / GRAD_ACCUM

        scaler.scale(loss).backward()
        model._feat_cache.clear()

        if (i + 1) % GRAD_ACCUM == 0:
            scaler.unscale_(opt)
            clip_all()
            scaler.step(opt)
            scaler.update()
            opt.zero_grad(set_to_none=True); sched.step(); gstep += 1

            if gstep % LOG_EVERY == 0:
                lv = float(o.loss.item())
                if not np.isfinite(lv):
                    print(f"\n!! L_VQA = {lv} pada step {gstep} — dihentikan.")
                    DIVERGED = True
                    break
                gg, gs = head.gate_mean_abs()
                log_line(TRAIN_LOG, {
                    "global_step": gstep, "epoch": ep,
                    "L_vqa": lv, "L_struct": float(rl.item()), "L_canon": float(lc),
                    "n_canon": int(n_lab), "gate_g": gg, "gate_s": gs,
                    "pe_scale": round(float(torch.tanh(head.pe_scale).item()), 4),
                    "gpu_gb": round(torch.cuda.memory_allocated() / 1e9, 2) if torch.cuda.is_available() else 0.0,
                    "elapsed_s": round(time.time() - t0, 1)
                })
                pbar.set_postfix(vqa=f"{lv:.3f}", canon=f"{float(lc):.3f}", g=f"{gg:.3f}")

            if gstep % VAL_EVERY == 0:
                validate_vqa(gstep, ep, max_batches=50)
            if gstep % SAVE_EVERY == 0:
                save_ckpt(gstep); save_resume(gstep, ep)

    if DIVERGED:
        break
    validate_vqa(gstep, ep, max_batches=100)
    save_ckpt(gstep, tag=f"epoch-{ep:02d}")
    save_resume(gstep, ep + 1)

if DIVERGED:
    CONFIG["status"] = "ABORTED - jangan dipakai"
    CONFIG["diverged_at_step"] = gstep
    (RUN_DIR / "config.json").write_text(json.dumps(CONFIG, indent=2))
    print("RUN DIHENTIKAN. Checkpoint `final` tidak ditulis.")
else:
    save_ckpt(gstep, tag="final")
    CONFIG["finished_at"] = datetime.now().isoformat(timespec="seconds")
    CONFIG["total_steps"] = gstep
    CONFIG["wall_clock_hours"] = round((time.time() - t0) / 3600, 2)
    CONFIG["status"] = "ok"
    (RUN_DIR / "config.json").write_text(json.dumps(CONFIG, indent=2))
    print(f"\n✓ Training {VARIANT} selesai dalam {CONFIG['wall_clock_hours']} jam -> {RUN_DIR}")


[VAL step 0 ep 0] val_L_VQA=5.0737 | M4 5.0737 (Δ +0.0000) | canon acc 10.0% | gate |g| 0.0000/0.0000 | pe_scale=0.0000


B1_M4_CanonBridge epoch 0:   0%|          | 0/8064 [00:00<?, ?it/s]


[VAL step 500 ep 0] val_L_VQA=2.6215 | M4 2.6206 (Δ +0.0009) | canon acc 70.8% | gate |g| 0.0161/0.0069 | pe_scale=0.0217
saved checkpoint -> step-000500

[VAL step 1000 ep 0] val_L_VQA=2.0209 | M4 2.0183 (Δ +0.0026) | canon acc 76.3% | gate |g| 0.0439/0.0129 | pe_scale=0.1650
saved checkpoint -> step-001000

[VAL step 1500 ep 0] val_L_VQA=1.6484 | M4 1.6534 (Δ -0.0050) | canon acc 79.2% | gate |g| 0.0602/0.0155 | pe_scale=0.2964
saved checkpoint -> step-001500

[VAL step 2000 ep 0] val_L_VQA=1.4155 | M4 1.4084 (Δ +0.0071) | canon acc 78.8% | gate |g| 0.0687/0.0193 | pe_scale=0.4092
saved checkpoint -> step-002000

[VAL step 2500 ep 0] val_L_VQA=1.2486 | M4 1.2526 (Δ -0.0040) | canon acc 79.7% | gate |g| 0.0751/0.0216 | pe_scale=0.4863
saved checkpoint -> step-002500

[VAL step 3000 ep 0] val_L_VQA=1.1603 | M4 1.1613 (Δ -0.0010) | canon acc 81.2% | gate |g| 0.0734/0.0238 | pe_scale=0.5335
saved checkpoint -> step-003000

[VAL step 3500 ep 0] val_L_VQA=1.0798 | M4 1.0835 (Δ -0.0037) | 

B1_M4_CanonBridge epoch 1:   0%|          | 0/8064 [00:00<?, ?it/s]


[VAL step 4500 ep 1] val_L_VQA=1.0067 | M4 1.0088 (Δ -0.0021) | canon acc 80.8% | gate |g| 0.0727/0.0287 | pe_scale=0.6240
saved checkpoint -> step-004500

[VAL step 5000 ep 1] val_L_VQA=0.9688 | M4 0.9782 (Δ -0.0094) | canon acc 81.7% | gate |g| 0.0722/0.0301 | pe_scale=0.6440
saved checkpoint -> step-005000

[VAL step 5500 ep 1] val_L_VQA=0.9425 | M4 0.9526 (Δ -0.0101) | canon acc 80.9% | gate |g| 0.0689/0.0322 | pe_scale=0.6619
saved checkpoint -> step-005500

[VAL step 6000 ep 1] val_L_VQA=0.9188 | M4 0.9333 (Δ -0.0145) | canon acc 81.8% | gate |g| 0.0712/0.0340 | pe_scale=0.6780
saved checkpoint -> step-006000

[VAL step 6500 ep 1] val_L_VQA=0.8983 | M4 0.9079 (Δ -0.0096) | canon acc 81.8% | gate |g| 0.0705/0.0351 | pe_scale=0.6922
saved checkpoint -> step-006500

[VAL step 7000 ep 1] val_L_VQA=0.8835 | M4 0.8946 (Δ -0.0111) | canon acc 81.6% | gate |g| 0.0703/0.0371 | pe_scale=0.7053
saved checkpoint -> step-007000

[VAL step 7500 ep 1] val_L_VQA=0.8593 | M4 0.8723 (Δ -0.0130) |

B1_M4_CanonBridge epoch 2:   0%|          | 0/8064 [00:00<?, ?it/s]


[VAL step 8500 ep 2] val_L_VQA=0.8414 | M4 0.8572 (Δ -0.0158) | canon acc 81.2% | gate |g| 0.0691/0.0396 | pe_scale=0.7375
saved checkpoint -> step-008500

[VAL step 9000 ep 2] val_L_VQA=0.8264 | M4 0.8362 (Δ -0.0098) | canon acc 82.0% | gate |g| 0.0713/0.0417 | pe_scale=0.7440
saved checkpoint -> step-009000

[VAL step 9500 ep 2] val_L_VQA=0.8148 | M4 0.8268 (Δ -0.0120) | canon acc 81.6% | gate |g| 0.0716/0.0430 | pe_scale=0.7522
saved checkpoint -> step-009500

[VAL step 10000 ep 2] val_L_VQA=0.8157 | M4 0.8227 (Δ -0.0070) | canon acc 81.9% | gate |g| 0.0704/0.0440 | pe_scale=0.7568
saved checkpoint -> step-010000

[VAL step 10500 ep 2] val_L_VQA=0.7906 | M4 0.8042 (Δ -0.0136) | canon acc 81.7% | gate |g| 0.0727/0.0452 | pe_scale=0.7622
saved checkpoint -> step-010500

[VAL step 11000 ep 2] val_L_VQA=0.7898 | M4 0.8012 (Δ -0.0114) | canon acc 82.8% | gate |g| 0.0727/0.0459 | pe_scale=0.7673
saved checkpoint -> step-011000

[VAL step 11500 ep 2] val_L_VQA=0.7820 | M4 0.7988 (Δ -0.016

B1_M4_CanonBridge epoch 3:   0%|          | 0/8064 [00:00<?, ?it/s]


[VAL step 12500 ep 3] val_L_VQA=0.7669 | M4 0.7780 (Δ -0.0111) | canon acc 82.6% | gate |g| 0.0711/0.0493 | pe_scale=0.7773
saved checkpoint -> step-012500

[VAL step 13000 ep 3] val_L_VQA=0.7620 | M4 0.7742 (Δ -0.0122) | canon acc 82.5% | gate |g| 0.0713/0.0495 | pe_scale=0.7801
saved checkpoint -> step-013000

[VAL step 13500 ep 3] val_L_VQA=0.7627 | M4 0.7729 (Δ -0.0102) | canon acc 82.1% | gate |g| 0.0716/0.0498 | pe_scale=0.7828
saved checkpoint -> step-013500

[VAL step 14000 ep 3] val_L_VQA=0.7570 | M4 0.7654 (Δ -0.0084) | canon acc 82.4% | gate |g| 0.0736/0.0508 | pe_scale=0.7846
saved checkpoint -> step-014000

[VAL step 14500 ep 3] val_L_VQA=0.7484 | M4 0.7589 (Δ -0.0105) | canon acc 83.2% | gate |g| 0.0731/0.0506 | pe_scale=0.7868
saved checkpoint -> step-014500

[VAL step 15000 ep 3] val_L_VQA=0.7468 | M4 0.7561 (Δ -0.0093) | canon acc 82.3% | gate |g| 0.0738/0.0519 | pe_scale=0.7875
saved checkpoint -> step-015000

[VAL step 15500 ep 3] val_L_VQA=0.7394 | M4 0.7479 (Δ -0.

B1_M4_CanonBridge epoch 4:   0%|          | 0/8064 [00:00<?, ?it/s]


[VAL step 16500 ep 4] val_L_VQA=0.7327 | M4 0.7434 (Δ -0.0107) | canon acc 82.3% | gate |g| 0.0724/0.0528 | pe_scale=0.7889
saved checkpoint -> step-016500

[VAL step 17000 ep 4] val_L_VQA=0.7317 | M4 0.7404 (Δ -0.0087) | canon acc 82.3% | gate |g| 0.0736/0.0530 | pe_scale=0.7895
saved checkpoint -> step-017000

[VAL step 17500 ep 4] val_L_VQA=0.7307 | M4 0.7380 (Δ -0.0073) | canon acc 82.0% | gate |g| 0.0735/0.0539 | pe_scale=0.7891
saved checkpoint -> step-017500

[VAL step 18000 ep 4] val_L_VQA=0.7303 | M4 0.7382 (Δ -0.0079) | canon acc 82.2% | gate |g| 0.0719/0.0543 | pe_scale=0.7890
saved checkpoint -> step-018000

[VAL step 18500 ep 4] val_L_VQA=0.7249 | M4 0.7340 (Δ -0.0091) | canon acc 82.5% | gate |g| 0.0727/0.0545 | pe_scale=0.7891
saved checkpoint -> step-018500

[VAL step 19000 ep 4] val_L_VQA=0.7236 | M4 0.7319 (Δ -0.0083) | canon acc 82.8% | gate |g| 0.0724/0.0544 | pe_scale=0.7891
saved checkpoint -> step-019000

[VAL step 19500 ep 4] val_L_VQA=0.7228 | M4 0.7306 (Δ -0.

# PART E — INFERENSI GENERATIF 15.955 BARIS TEST SET & PERBANDINGAN LIVE
<a id="part-e"></a>

Inferensi **identik notebook 01 `[E2]`** (model & kepala fp16, greedy, `max_new_tokens=64`, `attention_mask` TIDAK dikirim ke `.generate()`), sehingga selisih terhadap M4 bukan artefak decoding. Semua pembanding (M4, A0) dibaca dari file prediksinya — tidak ada angka acuan yang ditulis tangan.

## [E1] Muat Test Set & pilih run

In [14]:
# [E1] Muat Test Set (15.955 QA pairs) & Pilih Folder Run yang Akan Dievaluasi
from datasets import load_from_disk
te = load_from_disk(str(BASE / "test")).to_pandas()
print(f"✓ Total test rows loaded: {len(te)}")
assert len(te) == 15955

eval_cands = [p for p in sorted(OUTDIR.glob("*")) if p.is_dir() and (p / "checkpoints/final").exists()]
match_eval = [p for p in eval_cands if VARIANT in p.name]
TARGET_RUN = match_eval[-1] if match_eval else (eval_cands[-1] if eval_cands else globals().get("RUN_DIR", None))
assert TARGET_RUN is not None, f"Tidak ada run dengan checkpoints/final di {OUTDIR}"
print(f"✓ TARGET_RUN untuk evaluasi: {TARGET_RUN}")
EVAL_DIR = TARGET_RUN / "eval"
EVAL_DIR.mkdir(parents=True, exist_ok=True)

✓ Total test rows loaded: 15955
✓ TARGET_RUN untuk evaluasi: /content/drive/MyDrive/DOKTORAL/KvasirVQAX1/experiments/Paper2plus_Bridge_r32/20261103-050623_B1_M4_CanonBridge_r32_s42


## [E2] Load checkpoint final & jalankan inferensi

In [15]:
# [E2] Load Checkpoint Final (LoRA + GeoFiLMBridge + image_projection) & Jalankan Inferensi
from peft import PeftModel
from tqdm.auto import tqdm
import transformers.dynamic_module_utils as dmu
orig_get_imports = dmu.get_imports

GEN_CFG = {"max_new_tokens": 64}           # identik notebook 01 (greedy)

def _skip_flash_e(name, *args, **kwargs):
    if "flash_attn" in str(name): return None
    return orig_get_imports(name, *args, **kwargs)

def load_bridge_for_eval(run_dir):
    cfg = json.loads((run_dir / "config.json").read_text())
    ckpt = run_dir / "checkpoints/final"
    specs = cfg["canon_specs"]                            # DIPAKU dari run, bukan dihitung ulang
    assert cfg.get("core_version") == GB.CORE_VERSION, \
        f"Run dilatih dengan core {cfg.get('core_version')}, modul sekarang {GB.CORE_VERSION}"
    with patch("transformers.dynamic_module_utils.get_imports", _skip_flash_e):
        proc = AutoProcessor.from_pretrained(BASE_ID, trust_remote_code=True)
        base = AutoModelForCausalLM.from_pretrained(BASE_ID, trust_remote_code=True, torch_dtype=torch.float16)
    m = PeftModel.from_pretrained(base, str(ckpt)).to(dev).eval()
    core_e = _unwrap(m)
    ip_st = torch.load(ckpt / "image_proj.pt", map_location=dev)
    core_e.image_projection.data.copy_(ip_st["image_projection"].to(device=dev, dtype=core_e.image_projection.dtype))
    core_e.image_proj_norm.load_state_dict(ip_st["image_proj_norm"])

    h_mod = GeoFiLMBridge(core_e.image_projection.shape[1], specs, cfg["geo_kwargs"],
                          use_bridge=cfg["variant_cfg"]["use_bridge"])
    h_mod.load_state_dict(torch.load(ckpt / HEAD_FILE, map_location="cpu"))
    h_mod.to(device=dev, dtype=torch.float16).eval()

    cache = {"query_embeds": None, "query_mask": None, "canon_logits": None}
    orig_encode = core_e._encode_image
    def wrapped_encode(pixel_values):
        out = orig_encode(pixel_values)
        bridged, _, lg = h_mod.forward_bridge(out, query_embeds=cache.get("query_embeds"),
                                              query_mask=cache.get("query_mask"))
        cache["canon_logits"] = lg
        return bridged.to(out.dtype)
    core_e._encode_image = wrapped_encode
    m._feat_cache = cache
    print(f"✓ {run_dir.name} | variant {cfg['variant']} | {len(specs)} kategori kanonik | gerbang:",
          h_mod.gate_mean_abs())
    return m, proc, h_mod, core_e, specs

eval_model, eval_proc, eval_head, eval_core, EVAL_SPECS = load_bridge_for_eval(TARGET_RUN)
EVAL_LABELER = CanonLabeler(EVAL_SPECS)

PRED_PATH  = EVAL_DIR / "predictions.json"          # {idx: prosa} — format identik M1-M4 (notebook 02 siap)
CANON_PATH = EVAL_DIR / "canon_preds.json"          # {idx: {kategori: [atom]}} — diagnostik [E4]
predictions = {int(k): v for k, v in json.loads(PRED_PATH.read_text()).items()} if PRED_PATH.exists() else {}
canon_preds = {int(k): v for k, v in json.loads(CANON_PATH.read_text()).items()} if CANON_PATH.exists() else {}
todo = [i for i in range(len(te)) if i not in predictions]
print(f"Sudah ada {len(predictions)} | sisa {len(todo)}")

def resolve_image(img_id):
    stem = strip_jpg(img_id)
    cand = IMG_DIR / f"{stem}.jpg"
    if cand.exists(): return cand
    hit = list(IMG_DIR.glob(f"{stem}.*"))
    return hit[0] if hit else None

@torch.no_grad()
def run_generate_batch(prompts_list, imgs_list):
    inputs = eval_proc(text=prompts_list, images=imgs_list, return_tensors="pt", padding=True)
    inputs = {k: v.to(dev) for k, v in inputs.items()}
    inputs["pixel_values"] = inputs["pixel_values"].to(torch.float16)
    eval_model._feat_cache["query_embeds"] = eval_core.get_input_embeddings()(inputs["input_ids"]).detach()
    eval_model._feat_cache["query_mask"] = inputs.get("attention_mask")
    eval_model._feat_cache["canon_logits"] = None
    # JANGAN kirim attention_mask ke .generate(): Florence-2 menyusun mask 577+L sendiri
    gen_ids = eval_model.generate(input_ids=inputs["input_ids"], pixel_values=inputs["pixel_values"], **GEN_CFG)
    txt = [a.strip() for a in eval_proc.batch_decode(gen_ids, skip_special_tokens=True)]
    lg = eval_model._feat_cache.get("canon_logits")
    cp = canonical_predict(lg, EVAL_LABELER) if lg is not None else [dict() for _ in txt]
    return txt, cp

def _flush():
    PRED_PATH.write_text(json.dumps({str(k): predictions[k] for k in sorted(predictions)}, indent=2))
    CANON_PATH.write_text(json.dumps({str(k): canon_preds[k] for k in sorted(canon_preds)}))

def _to_names(cp):
    inv = {}
    for c in EVAL_LABELER.cats:
        inv[c] = EVAL_SPECS[c]["atoms"]
    return {c: sorted(inv[c][j] for j in js) for c, js in cp.items()}

BATCH_INF = 16 if (torch.cuda.is_available() and torch.cuda.get_device_properties(0).total_memory / 1e9 >= 30.0) else 8
for s in tqdm(range(0, len(todo), BATCH_INF), desc=f"Infer {TARGET_RUN.name[:28]}"):
    idxs = todo[s:s + BATCH_INF]
    imgs, prompts, valid = [], [], []
    for idx in idxs:
        row = te.iloc[idx]
        p_img = resolve_image(row["img_id"])
        if p_img is None:
            predictions[idx] = ""; continue
        imgs.append(Image.open(p_img).convert("RGB"))
        prompts.append(PROMPT_PREFIX + str(row["question"]).strip())
        valid.append(idx)
    if not valid:
        continue
    try:
        txt, cp = run_generate_batch(prompts, imgs)
    except RuntimeError:
        torch.cuda.empty_cache()
        txt, cp = [], []
        for p_txt, im in zip(prompts, imgs):
            t1, c1 = run_generate_batch([p_txt], [im]); txt += t1; cp += c1
    for idx, t, c in zip(valid, txt, cp):
        predictions[idx] = t
        canon_preds[idx] = _to_names(c)
    if (s // BATCH_INF) % 25 == 0:
        _flush()
_flush()
n_empty = sum(1 for v in predictions.values() if not str(v).strip())
print(f"\n✓ Inferensi selesai: {len(predictions)} baris ({n_empty} kosong) -> {PRED_PATH}")
assert len(predictions) == len(te)

✓ 20261103-050623_B1_M4_CanonBridge_r32_s42 | variant B1_M4_CanonBridge | 16 kategori kanonik | gerbang: (0.07306, 0.05484)
Sudah ada 0 | sisa 15955


Infer 20261103-050623_B1_M4_CanonB:   0%|          | 0/998 [00:00<?, ?it/s]


✓ Inferensi selesai: 15955 baris (0 kosong) -> /content/drive/MyDrive/DOKTORAL/KvasirVQAX1/experiments/Paper2plus_Bridge_r32/20261103-050623_B1_M4_CanonBridge_r32_s42/eval/predictions.json


## [E3] Metrik live vs M4 & A0 + McNemar

In [16]:
# [E3] Token-F1 (fungsi PERSIS notebook 01, \w+) vs M4 & A0 dibaca LIVE + McNemar berpasangan
# Catatan: angka "68,29" milik M4 dihitung dengan fungsi ini. Notebook 03 memakai split() dan
# membandingkannya dengan 68,29 -> selisih palsu -6,34pp (yang benar -2,74pp). Di sini semua
# model dihitung ulang dengan fungsi yang sama dari file prediksinya.
from collections import Counter
from scipy.stats import binomtest

def token_f1(pred, gold):
    p_toks = re.findall(r"\w+", str(pred).lower())
    g_toks = re.findall(r"\w+", str(gold).lower())
    if not p_toks or not g_toks:
        return float(p_toks == g_toks)
    common = sum((Counter(p_toks) & Counter(g_toks)).values())
    if common == 0:
        return 0.0
    prec, rec = common / len(p_toks), common / len(g_toks)
    return 2 * prec * rec / (prec + rec)

def load_pred_dict(path):
    raw = json.loads(Path(path).read_text())
    if isinstance(raw, dict):
        return {int(k): str(v) for k, v in raw.items()}
    return {int(r.get("idx", i)): str(r.get("pred", r.get("pred_full", ""))) for i, r in enumerate(raw)}

REFS = {}
_m4 = [p for p in sorted(M4_DIR.glob("*M4_GeoFiLM*")) if (p / "eval/predictions.json").exists()]
if _m4:
    REFS["M4_GeoFiLM"] = load_pred_dict(_m4[-1] / "eval/predictions.json")
if (A0_R32_DIR / "eval/predictions.json").exists():
    REFS["A0_VQAonly"] = load_pred_dict(A0_R32_DIR / "eval/predictions.json")

gold = te["answer"].astype(str).tolist()
cats = te["question_class"].map(as_class_list)
f1 = {"OURS": np.array([token_f1(predictions.get(i, ""), gold[i]) for i in range(len(te))])}
for k, d in REFS.items():
    f1[k] = np.array([token_f1(d.get(i, ""), gold[i]) for i in range(len(te))])

print("=" * 90)
print(f" TOKEN-F1 OVERALL (15.955) — {TARGET_RUN.name}")
print("=" * 90)
for k, v in f1.items():
    extra = f"   Δ OURS-{k}: {100*(f1['OURS'].mean()-v.mean()):+.2f}pp" if k != "OURS" else ""
    print(f"  {k:12s} {100*v.mean():6.2f}%{extra}")

if "complexity" in te.columns:
    print("\nPer kompleksitas:")
    for c in sorted(te["complexity"].unique()):
        msk = (te["complexity"] == c).to_numpy()
        line = f"  c{c} n={msk.sum():5d} | OURS {100*f1['OURS'][msk].mean():6.2f}"
        for k in REFS:
            line += f" | {k} {100*f1[k][msk].mean():6.2f} (Δ {100*(f1['OURS'][msk].mean()-f1[k][msk].mean()):+.2f})"
        print(line)

print("\nPer kategori (baris yang menanyakan kategori itu):")
rows = []
for c in sorted({x for L in cats for x in L}):
    msk = cats.map(lambda L: c in L).to_numpy()
    r = {"kategori": c, "n": int(msk.sum()), "OURS": round(100 * f1["OURS"][msk].mean(), 2)}
    for k in REFS:
        r[k] = round(100 * f1[k][msk].mean(), 2)
        r[f"Δ_{k}"] = round(r["OURS"] - r[k], 2)
    rows.append(r)
df_cat = pd.DataFrame(rows).sort_values(f"Δ_M4_GeoFiLM" if "M4_GeoFiLM" in REFS else "OURS")
display(df_cat)
df_cat.to_csv(EVAL_DIR / "token_f1_per_category.csv", index=False)

print("\nMcNemar berpasangan (token-F1 >= 0,5 dianggap benar):")
mc = []
for k in REFS:
    a, b = (f1["OURS"] >= 0.5).astype(int), (f1[k] >= 0.5).astype(int)
    n01, n10 = int(((a == 1) & (b == 0)).sum()), int(((a == 0) & (b == 1)).sum())
    p = binomtest(n01, n01 + n10, 0.5).pvalue if (n01 + n10) else 1.0
    mc.append({"vs": k, "fix": n01, "break": n10, "delta_pp": round(100 * (a.mean() - b.mean()), 2), "p": p})
    print(f"  vs {k:12s} fix {n01:5d} | break {n10:5d} | Δ {100*(a.mean()-b.mean()):+.2f}pp | p={p:.3e}")
pd.DataFrame(mc).to_csv(EVAL_DIR / "mcnemar_tokenf1.csv", index=False)
print("\nCatatan: klaim utama tetap Dual-LLM Judge (notebook 02 mendeteksi run ini otomatis).")

 TOKEN-F1 OVERALL (15.955) — 20261103-050623_B1_M4_CanonBridge_r32_s42
  OURS          68.90%
  M4_GeoFiLM    68.29%   Δ OURS-M4_GeoFiLM: +0.60pp
  A0_VQAonly    65.64%   Δ OURS-A0_VQAonly: +3.25pp

Per kompleksitas:
  c1 n= 5496 | OURS  71.77 | M4_GeoFiLM  71.46 (Δ +0.31) | A0_VQAonly  68.68 (Δ +3.09)
  c2 n= 5251 | OURS  64.46 | M4_GeoFiLM  63.65 (Δ +0.81) | A0_VQAonly  60.50 (Δ +3.96)
  c3 n= 5208 | OURS  70.35 | M4_GeoFiLM  69.64 (Δ +0.71) | A0_VQAonly  67.63 (Δ +2.72)

Per kategori (baris yang menanyakan kategori itu):


,kategori,n,OURS,M4_GeoFiLM,Δ_M4_GeoFiLM,A0_VQAonly,Δ_A0_VQAonly
5,finding_presence,237,76.54,77.20,-0.66,78.78,-2.24
9,landmark_color,55,58.34,58.66,-0.32,57.80,0.54
16,procedure_type,2303,68.78,68.77,0.01,66.61,2.17
13,polyp_removal_status,2312,67.07,66.94,0.13,62.81,4.26
15,polyp_type,1944,64.96,64.79,0.17,60.98,3.98
10,landmark_location,1224,67.80,67.54,0.26,64.07,3.73
11,landmark_presence,1264,71.69,71.41,0.28,69.11,2.58
12,polyp_count,1900,68.52,68.14,0.38,65.53,2.99
8,instrument_presence,1837,66.98,66.48,0.50,63.23,3.75
3,box_artifact_presence,2319,75.38,74.70,0.68,73.44,1.94



McNemar berpasangan (token-F1 >= 0,5 dianggap benar):
  vs M4_GeoFiLM   fix   626 | break   492 | Δ +0.84pp | p=6.831e-05
  vs A0_VQAonly   fix  1335 | break   694 | Δ +4.02pp | p=1.425e-46

Catatan: klaim utama tetap Dual-LLM Judge (notebook 02 mendeteksi run ini otomatis).


## [E4] Diagnostik kepala kanonik & gerbang

In [17]:
# [E4] Diagnostik — akurasi kepala kanonik di TEST per kategori & bukaan gerbang (BUKAN klaim utama)
# Tujuan: memastikan bridge benar-benar dipakai (gerbang terbuka) dan kepala memprediksi dengan
# baik. Bila kepala akurat tetapi token-F1 kategori itu tidak naik, celahnya ada di decoder.
canon_preds = {int(k): v for k, v in json.loads(CANON_PATH.read_text()).items()}
cnt = {c: [0, 0] for c in EVAL_LABELER.cats}
for i, (q, o) in enumerate(zip(te["question_class"], te["original"])):
    for ci, idx in EVAL_LABELER.encode(q, o):
        c = EVAL_LABELER.cats[ci]
        s0 = EVAL_LABELER.slices[ci][0]
        gold_atoms = {EVAL_SPECS[c]["atoms"][j - s0] for j in idx}
        pred_atoms = set(canon_preds.get(i, {}).get(c, []))
        cnt[c][0] += 1; cnt[c][1] += int(gold_atoms == pred_atoms)
gsum = eval_head.gate_summary()
rows = []
for c, (n, ok) in cnt.items():
    gg, gs = gsum.get(c, (0.0, 0.0))
    rows.append({"kategori": c, "n_test": n, "kepala_exact_%": round(100 * ok / max(n, 1), 2),
                 "gate_global": gg, "gate_spatial": gs, "multi": EVAL_SPECS[c]["multi"]})
df_head = pd.DataFrame(rows).sort_values("kepala_exact_%")
display(df_head)
df_head.to_csv(EVAL_DIR / "canon_head_accuracy.csv", index=False)
print("Gerbang |tanh(g)| rata-rata (global, spasial):", eval_head.gate_mean_abs())

,kategori,n_test,kepala_exact_%,gate_global,gate_spatial,multi
0,abnormality_color,1862,49.46,0.0815,-0.0599,True
8,landmark_location,1224,55.15,0.1412,0.0124,True
1,abnormality_location,1897,62.84,0.0339,-0.1100,True
6,instrument_location,1913,69.26,0.0927,-0.1331,True
12,polyp_size,1925,72.73,0.0714,0.0854,True
13,polyp_type,1944,73.92,0.1230,-0.0186,True
7,instrument_presence,1837,90.75,0.0751,-0.1055,True
9,landmark_presence,1260,92.70,0.0750,0.0129,False
4,finding_count,2108,93.12,0.0444,-0.0652,False
3,box_artifact_presence,2319,97.02,0.0195,-0.0585,False


Gerbang |tanh(g)| rata-rata (global, spasial): (0.07306, 0.05484)
